In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from glm.extras.participation_intensity import (
    CoverageSimulation,
    ParameterSampleRanges,
    ParticipationIntensityMean,
    ParticipationIntensityModel,
    sample_data,
)

# Participation–Intensity Mean

In [ ]:
mean = ParticipationIntensityMean(-1.0, 5.0, -0.5, -2.0)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 3.0, num=250)
ax.plot(xs, mean(xs))
ax.axvline(mean.argmax(), linestyle='--', color='grey')
ax.scatter(mean.argmax(), mean.max(), s=125)
ax.set_title(str(mean))

## Sample Parameters

In [ ]:
def sample_means(ranges: ParameterSampleRanges, n: int) -> list[ParticipationIntensityMean]:
    means: list[ParticipationIntensityMean] = []
    for i in range(n):
        a = np.random.uniform(low=ranges.a[0], high=ranges.a[1])
        b = np.random.uniform(low=ranges.b[0], high=ranges.b[1])
        α = np.random.uniform(low=ranges.α[0], high=ranges.α[1])
        β = np.random.uniform(low=ranges.β[0], high=ranges.β[1])
        means.append(ParticipationIntensityMean(a, b, α, β))
    return means

In [ ]:
rgs = ParameterSampleRanges(
    a=(-0.1, 0.0),
    b=(0.5, 0.6),
    α=(-1.0, -0.25),
    β=(-2.0, -0.1)
)
means = sample_means(ranges=rgs, n=10)
means = list(sorted(means, key=lambda m: -m.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xmax = max(m.argmax() for m in means) + 1.0
xs = np.linspace(0.0, xmax, num=250)

cmap = plt.get_cmap("viridis")
colors = cmap(np.linspace(0, 1, len(means)))

for m, color in zip(means, colors):
    ax.plot(xs, m(xs), color=color)
    ax.scatter(m.argmax(), m.max(), s=125, color=color)
    ax.axvline(m.argmax(), linestyle='--', color='grey', alpha=0.25)
ax.set_title("Various Participation–Intensity Means with Maxima")

# Sample Dataset

In [ ]:
xs = np.linspace(0.0, 3.0, num=1000)

mean = ParticipationIntensityMean(-1.0, 5.0, -0.5, -2.0)
X = sample_data(mean, xs)
model = ParticipationIntensityModel().fit(X)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 3.0, num=250)
ax.plot(xs, mean(xs))
ax.scatter(mean.argmax(), mean.max(), s=125)
ax.axvline(mean.argmax(), linestyle='--')

ax.plot(xs, model.mean_(xs), linestyle=':', color='orange')
ax.scatter(model.mean_.argmax(), model.mean_.max(), s=75, color='orange')
ax.axvline(model.mean_.argmax(), linestyle='--', color='orange')
ax.axvspan(
    model.argmax_() - 1.96 * model.argmax_se_(),
    model.argmax_() + 1.96 * model.argmax_se_(),
    color='orange',
    alpha=0.25,
)

ax.set_title("Participation–Intensity Mean with Fit from Sampled Data")

In [ ]:
mean

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

xs = np.linspace(0.0, 3.0, num=250)
ax.plot(xs, mean(xs))
ax.scatter(mean.argmax(), mean.max(), s=125)
ax.axvline(mean.argmax(), linestyle='--')

for _ in range(100):
    X = sample_data(mean, xs)
    model = ParticipationIntensityModel().fit(X)
    ax.plot(xs, model.mean_(xs), linestyle=':', color='orange', alpha=0.25)
    ax.scatter(model.mean_.argmax(), model.mean_.max(), s=75, color='orange', alpha=0.75)
    ax.axvline(model.mean_.argmax(), linestyle='--', color='orange', alpha=0.75)

ax.set_title("Participation–Intensity Mean with Repeated Fits from Sampled Data")

In [ ]:
ams: list[float] = []
xs = np.linspace(0.0, 3.0, num=1000)

for _ in range(1000):
    X = sample_data(mean, xs)
    model = ParticipationIntensityModel().fit(X)
    ams.append(model.argmax_())
ams = np.asarray(ams)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
_  = ax.hist(ams, bins=100)
ax.axvline(ams.mean(), linestyle='--', linewidth=3, color='black')

# Coverage

In [ ]:
sim = CoverageSimulation(mean).sample(n=5000, nxs=100_000)

In [ ]:
cis = sim.argmax_cis_()

In [ ]:
within = (cis[:, 0] <= mean.argmax()) & (mean.argmax() <= cis[:, 1])
within.mean()

In [ ]:
def moneys(total: float) -> float:
    return (total * (1 - 0.37) - 50.0) / 7.0

moneys(700)